# 🔗 04 — RAG + BERT Entegrasyonu: İstanbul Chatbot

## Mimari
1. **Kullanıcı sorusu** → BERT intent detection
2. **RAG** → FAISS'ten en ilgili belge parçaları
3. **Prompt Builder** → Soru + bağlam birleştirilir
4. **Yanıt** → Belge tabanlı cevap üretilir

In [ ]:
!pip install transformers torch sentence-transformers faiss-cpu gradio -q
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'data'))
print('✅ Hazir!')

## 📥 Adım 1: Model ve Verileri Yukle

In [ ]:
import torch
import faiss
import pickle
from transformers import BertTokenizer, BertForSequenceClassification
from sentence_transformers import SentenceTransformer
from data.intent_dataset import INTENT_LABELS

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ON_CPU = (device.type == 'cpu')
print(f'Cihaz: {device}')

print('BERT Intent modeli yukleniyor...')
try:
    intent_tokenizer = BertTokenizer.from_pretrained('model/istanbul_intent')
    intent_model = BertForSequenceClassification.from_pretrained('model/istanbul_intent').to(device)
    intent_model.eval()
    print('✅ Fine-tuned model yuklendi!')
except Exception as e:
    print(f'Fine-tuned model bulunamadi ({e}), base model kullaniliyor...')
    intent_tokenizer = BertTokenizer.from_pretrained('dbmdz/bert-base-turkish-cased')
    intent_model = BertForSequenceClassification.from_pretrained(
        'dbmdz/bert-base-turkish-cased', num_labels=len(INTENT_LABELS),
        id2label={i:l for i,l in enumerate(INTENT_LABELS)},
        label2id={l:i for i,l in enumerate(INTENT_LABELS)}
    ).to(device)

print('\nRAG bileşenleri yukleniyor...')
embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
faiss_index = faiss.read_index('rag_data/istanbul_index.faiss')
with open('rag_data/chunks.pkl', 'rb') as f:
    rag_data = pickle.load(f)
    all_chunks = rag_data['chunks']
    chunk_meta = rag_data['meta']
print(f'✅ FAISS indeksi: {faiss_index.ntotal} vektor')

## ⚙️ Adım 2: Chatbot Fonksiyonları

In [ ]:
def detect_intent(text):
    enc = intent_tokenizer(text, return_tensors='pt', truncation=True, max_length=128, padding='max_length').to(device)
    with torch.no_grad():
        logits = intent_model(**enc).logits[0]
    probs = torch.softmax(logits, dim=0).cpu().numpy()
    best_idx = probs.argmax()
    return INTENT_LABELS[best_idx], float(probs[best_idx])

def retrieve_context(query, k=3):
    q_vec = embed_model.encode([query]).astype('float32')
    distances, indices = faiss_index.search(q_vec, k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        results.append({'metin': all_chunks[idx], 'kaynak': chunk_meta[idx]['kaynak'], 'dist': dist})
    return results

print('✅ Fonksiyonlar tanimlandı!')

## 🧪 Adım 3: Tek Soru Testi

In [ ]:
def sor(soru):
    intent, score = detect_intent(soru)
    retrieved = retrieve_context(soru, k=2)
    kaynaklar = list({r['kaynak'] for r in retrieved})
    
    print(f'❓ Soru       : {soru}')
    print(f'🎯 Intent     : {intent} (%{score*100:.1f})')
    print(f'📚 Kaynaklar  : {", ".join(kaynaklar)}')
    print(f'\n📖 Bulunan bilgiler:')
    for i, r in enumerate(retrieved):
        print(f'  [{i+1}] {r["metin"][:200]}...')
    print()

sor('Ayasofya hangi yüzyılda yapıldı?')
sor('İstanbul metrosu nasıl kullanılır?')
sor('İstanbul\'da ne tür yemekler yenir?')

## 🖥️ Adım 4: Gradio Arayüzü (Chatbot)

In [ ]:
import gradio as gr

INTENT_EMOJIS = {
    'tarih_ve_kultur': '🏛️',
    'ulasim_ve_seyahat': '🚇',
    'yemek_ve_gastronomi': '🍽️',
    'cografya_ve_semt': '🗺️',
    'egitim_ve_is': '🎓',
    'spor_ve_eglence': '⚽',
}

def chatbot_logic(message, history):
    intent, intent_score = detect_intent(message)
    retrieved = retrieve_context(message, k=3)
    kaynaklar = set([r['kaynak'] for r in retrieved])
    kaynak_str = ', '.join(kaynaklar).replace('_', ' ').title()
    emoji = INTENT_EMOJIS.get(intent, '🔍')
    
    cevap = f"{emoji} **Konu:** `{intent.replace('_', ' ').title()}` *(Güven: %{intent_score*100:.1f})*\n\n"
    cevap += f"📚 **Kaynak:** {kaynak_str}\n\n---\n\n"
    
    for r in retrieved[:2]:
        cevap += f"📍 {r['metin'][:350]}\n\n"
    
    cevap += "---\n*Başka bir sorunuz varsa sormaktan çekinmeyin! 🕌*"
    return cevap

with gr.Blocks(theme=gr.themes.Soft(), title='İstanbul Asistan') as demo:
    gr.HTML("<h1 style='text-align:center;color:#c0392b'>🕌 İstanbul Şehir Rehberi</h1>")
    gr.HTML("<p style='text-align:center;color:#7f8c8d'>RAG + BERT destekli akıllı İstanbul asistanı</p>")
    gr.ChatInterface(
        fn=chatbot_logic,
        chatbot=gr.Chatbot(height=500),
        textbox=gr.Textbox(placeholder='İstanbul hakkında bir şeyler sorun...', container=False, scale=7),
        examples=[
            'Ayasofya nerede ve ne zaman yapıldı?',
            'İstanbul metrosu nasıl kullanılır?',
            'İstanbul\'da en iyi yemekler nelerdir?',
            'Boğaz Köprüleri hangileri?',
            'Galatasaray UEFA kupasını aldı mı?',
        ],
    )

if __name__ == '__main__':
    demo.launch(share=False, server_port=7860)